In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Import all csv files and convert to datetime
data_PressureLevel_raw = pd.read_csv('Data/Pressure Level Data.csv')
data_CloudandRain_raw = pd.read_csv('Data/Cloud Coverage and Precipitation.csv')
data_weather_raw = pd.read_csv('Data/Merged Weather.csv',index_col=0)
data_mt_raw = pd.read_csv('Data/everest_reduced.csv')

data_PressureLevel_raw['valid_time'] = pd.to_datetime(data_PressureLevel_raw['valid_time'], errors = 'coerce',format ='mixed')
data_CloudandRain_raw['valid_time'] = pd.to_datetime(data_CloudandRain_raw['valid_time'], errors = 'coerce',format='mixed')


data_weather_raw['valid_time'] = pd.to_datetime(data_weather_raw['valid_time'], errors = 'coerce', format = 'mixed')
data_mt_raw['smtdate'] = pd.to_datetime(data_mt_raw['smtdate'], errors= 'coerce', format = 'mixed')

In [3]:
data_PL = data_PressureLevel_raw.copy().sort_values(by = 'valid_time').reset_index(drop = True)
data_CR = data_CloudandRain_raw.copy().sort_values(by = 'valid_time').reset_index(drop = True)
data_Weather_Merged = data_weather_raw.copy().sort_values(by = 'valid_time').reset_index(drop = True)
data_mnt = data_mt_raw.copy().sort_values(by = 'smtdate').reset_index(drop = True)

In [4]:
# drop useless columns latitude and longitude
data_PL = data_PL.drop(columns = ['latitude','longitude'])
data_CR = data_CR.drop(columns = ['latitude','longitude'])

# remove unwatned pressure level data
Selected_Pressure_Levels = [300,400] # will select all pressure levels between the 2 numbers

Pressure_Level_mask = data_PL['pressureLevel'].between(Selected_Pressure_Levels[0],Selected_Pressure_Levels[1])
data_PL = data_PL[Pressure_Level_mask].reset_index(drop = True)

In [5]:
Data_All = {
    'Pressure Level': data_PL,
    'Cloud Rain': data_CR,
    'Merged': data_Weather_Merged,
    'Mountain': data_mnt

}

df_info = {}

for name,df in Data_All.items():
    df_info[name] = {
        "columns": df.columns.tolist(),
        "shape": df.shape,
        "rows": len(df),
        "num_columns": len(df.columns)
    }

df_info = pd.DataFrame(df_info)
print(df_info)

                                         Pressure Level  \
columns      [valid_time, pressureLevel, z, u, v, r, t]   
shape                                        (91560, 7)   
rows                                              91560   
num_columns                                           7   

                        Cloud Rain  \
columns      [valid_time, tcc, tp]   
shape                  (274680, 3)   
rows                        274680   
num_columns                      3   

                                                        Merged  \
columns      [valid_time, tcc, tp, z_300, z_400, r_300, r_4...   
shape                                              (45780, 11)   
rows                                                     45780   
num_columns                                                 11   

                                                      Mountain  
columns      [expid, year, smtdate, host, totmembers, tothi...  
shape                                               

In [6]:
for name, df in df_info.items():
    print(f'{name}: {df.columns}')
    print('-'*30)

Pressure Level: ['valid_time', 'pressureLevel', 'z', 'u', 'v', 'r', 't']
------------------------------
Cloud Rain: ['valid_time', 'tcc', 'tp']
------------------------------
Merged: ['valid_time', 'tcc', 'tp', 'z_300', 'z_400', 'r_300', 'r_400', 't_300', 't_400', 'max_wind_speed_300', 'max_wind_speed_400']
------------------------------
Mountain: ['expid', 'year', 'smtdate', 'host', 'totmembers', 'tothired', 'success1', 'termreason', 'smtmembers']
------------------------------


Notes: 
* u and v unless (wind components)
* lat and long unless
* success1 think is unless? 



In [7]:
# Extracts dates ( day of summit and 1 other day dependent on step)
dates = []
step = -1 # date + step [days] ( date -1 day)

for date in data_mnt['smtdate']:
    dates.append(date + pd.Timedelta(days = step))
    dates.append(date)
dates = pd.DataFrame(dates)

In [8]:
# create and apply date masks

time_mask_Pressure_Level = data_PL['valid_time'].dt.normalize().isin(
    dates.iloc[:,0].dt.normalize())

time_mask_Cloud_Rain = data_CR['valid_time'].dt.normalize().isin(
    dates.iloc[:,0].dt.normalize())

time_mask_Merged = data_Weather_Merged['valid_time'].dt.normalize().isin(
    dates.iloc[:,0].dt.normalize())

data_PL = data_PL.loc[time_mask_Pressure_Level,:].reset_index(drop=True)
data_CR = data_CR.loc[time_mask_Cloud_Rain,:].reset_index(drop=True)
data_Weather_Merged = data_Weather_Merged.loc[time_mask_Merged,:].reset_index(drop=True)

In [9]:
# get daily rainfall/ precipitation (date filter already applied)
daily_rainfall = ( data_CR.groupby(data_CR['valid_time'].dt.date)['tp'].sum().reset_index() )

daily_rainfall.columns = ['date', 'total_rainfall']
print(daily_rainfall.info())
print(daily_rainfall.head(5))


<class 'pandas.DataFrame'>
RangeIndex: 543 entries, 0 to 542
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            543 non-null    object 
 1   total_rainfall  543 non-null    float64
dtypes: float64(1), object(1)
memory usage: 8.6+ KB
None
         date  total_rainfall
0  1995-05-10        0.000558
1  1995-05-11        0.000242
2  1995-05-12        0.000326
3  1995-05-13        0.000606
4  1995-05-14        0.001482


In [10]:
# Member Success as a percentage
data_mnt['%_mem_succs'] = data_mnt['smtmembers'] / data_mnt['totmembers']


In [11]:
# Aggregate daily weather data

daily_weather = data_Weather_Merged.groupby(data_Weather_Merged['valid_time'].dt.date).agg(
    wind300_max = ('max_wind_speed_300', 'max'),
    wind300_min = ('max_wind_speed_300', 'min'),
    wind300_mean = ('max_wind_speed_300', 'mean'),

    wind400_max = ('max_wind_speed_400', 'max'),
    wind400_min = ('max_wind_speed_400', 'min'),
    wind400_mean = ('max_wind_speed_400', 'mean'),
    
    temp300_max = ('t_300', 'max'),
    temp300_min = ('t_300', 'min'),
    temp300_mean = ('t_300', 'mean'),

    temp400_max = ('t_400', 'max'),
    temp400_min = ('t_400', 'min'),
    temp400_mean = ('t_400', 'mean'),

    z300_min = ('z_300', 'min'),
    z300_max = ('z_300', 'max'),

    z400_min = ('z_400', 'min'),
    z400_max = ('z_400', 'max'),
    
    r300_mean = ('r_300', 'mean'),
    r400_mean = ('r_400','mean'),

    tcc_max = ('tcc', 'max'),
    tcc_min = ('tcc','min'),
    tcc_mean = ('tcc','mean')
).reset_index()

daily_weather['valid_time'] = pd.to_datetime(daily_weather['valid_time'])
daily_rainfall['date'] = pd.to_datetime(daily_rainfall['date'])

daily_weather = pd.merge(
    daily_weather,
    daily_rainfall,
    left_on='valid_time',
    right_on='date',
    how='left'
).drop(columns=['date']).rename(columns={'total_rainfall': 'daily_rainfall'})

In [12]:
print(daily_weather.head(5))

  valid_time  wind300_max  wind300_min  wind300_mean  wind400_max  \
0 1995-05-10    15.848718     6.801484     11.543892     8.426801   
1 1995-05-11     5.666456     2.984347      4.294215     3.168471   
2 1995-05-12     6.885866     3.284288      5.473119     4.154729   
3 1995-05-13     9.480625     5.097534      6.821812     6.846526   
4 1995-05-14     5.135309     2.575506      4.099943     7.995140   

   wind400_min  wind400_mean  temp300_max  temp300_min  temp300_mean  ...  \
0     3.369340      5.402568    243.91824    242.01999    243.122118  ...   
1     1.435653      2.578355    246.19069    245.05974    245.411988  ...   
2     3.141184      3.486980    247.03448    246.01244    246.391160  ...   
3     1.069923      3.162124    247.43263    246.99063    247.175823  ...   
4     1.383607      4.692359    247.22226    245.95332    246.744780  ...   

   z300_min   z300_max   z400_min  z400_max  r300_mean  r400_mean   tcc_max  \
0  94397.19  94836.375  73830.090  74194.73

In [13]:
print(data_mnt.head(5))
data_mnt.info()


       expid  year    smtdate  host  totmembers  tothired  success1  \
0  EVER95106  1995 1995-05-11     2          23         2      True   
1  EVER95103  1995 1995-05-12     2          13         7      True   
2  EVER95105  1995 1995-05-12     2           6         0      True   
3  EVER95113  1995 1995-05-12     2          12         4      True   
4  EVER95109  1995 1995-05-12     1          11        15     False   

   termreason  smtmembers  %_mem_succs  
0           1           7     0.304348  
1           1           2     0.153846  
2           1           3     0.500000  
3           1           8     0.666667  
4           5           0     0.000000  
<class 'pandas.DataFrame'>
RangeIndex: 1611 entries, 0 to 1610
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   expid        1611 non-null   str           
 1   year         1611 non-null   int64         
 2   smtdate      1611 non-null   d

In [14]:
Data_Merged = pd.merge(
    daily_weather,
    data_mnt,
    left_on= 'valid_time',
    right_on='smtdate',
    how = 'inner'
)

In [15]:
Data_Merged = Data_Merged.drop(columns = ['success1','smtdate','year'])

print(Data_Merged.info())
Data_Merged.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1611 entries, 0 to 1610
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   valid_time      1611 non-null   datetime64[s]
 1   wind300_max     1611 non-null   float64      
 2   wind300_min     1611 non-null   float64      
 3   wind300_mean    1611 non-null   float64      
 4   wind400_max     1611 non-null   float64      
 5   wind400_min     1611 non-null   float64      
 6   wind400_mean    1611 non-null   float64      
 7   temp300_max     1611 non-null   float64      
 8   temp300_min     1611 non-null   float64      
 9   temp300_mean    1611 non-null   float64      
 10  temp400_max     1611 non-null   float64      
 11  temp400_min     1611 non-null   float64      
 12  temp400_mean    1611 non-null   float64      
 13  z300_min        1611 non-null   float64      
 14  z300_max        1611 non-null   float64      
 15  z400_min        1611 non-null   

,valid_time,wind300_max,wind300_min,wind300_mean,wind400_max,wind400_min,wind400_mean,temp300_max,temp300_min,temp300_mean,...,tcc_min,tcc_mean,daily_rainfall,expid,host,totmembers,tothired,termreason,smtmembers,%_mem_succs
0,1995-05-11,5.666456,2.984347,4.294215,3.168471,1.435653,2.578355,246.19069,245.05974,245.411988,...,0.040680,0.561104,0.000242,EVER95106,2,23,2,1,7,0.304348
1,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.063324,0.600616,0.000326,EVER95103,2,13,7,1,2,0.153846
2,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.063324,0.600616,0.000326,EVER95105,2,6,0,1,3,0.500000
3,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.063324,0.600616,0.000326,EVER95113,2,12,4,1,8,0.666667
4,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.063324,0.600616,0.000326,EVER95109,1,11,15,5,0,0.000000
5,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.707428,0.898720,0.001482,EVER95114,2,19,12,1,9,0.473684
6,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.707428,0.898720,0.001482,EVER95110,2,10,3,1,1,0.100000
7,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.707428,0.898720,0.001482,EVER95111,2,8,2,1,3,0.375000
8,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.707428,0.898720,0.001482,EVER95104,2,20,14,1,8,0.400000
9,1995-05-15,7.593754,4.862681,5.913409,7.640121,0.979840,5.055960,246.86212,245.58328,246.334735,...,0.785858,0.921677,0.002844,EVER95112,1,12,9,1,2,0.166667


Next steps
* add 1 previous day weather

clean mnt data? 

Split Data
* remove termreason, smtmembers, %_mem_succs, expid
* use split function (90/10)
* Export to 2 csv files (train/val and test)

In [16]:
Data_Merged = Data_Merged.drop(columns = ['termreason','expid','smtmembers'])


prev_rain_lookup = daily_weather.set_index('valid_time')['daily_rainfall']
Data_Merged['prev_day_rainfall'] = (Data_Merged['valid_time'] - pd.Timedelta(days=1)).map(prev_rain_lookup)

In [17]:
from sklearn.model_selection import train_test_split

X = Data_Merged.drop(columns=['%_mem_succs'])
y = Data_Merged['%_mem_succs']

Xtrain, Xtest, ytrain, ytest = train_test_split( X,y,test_size=0.1,shuffle= False)

Train = pd.concat([Xtrain,ytrain],axis = 1)
Test = pd.concat([Xtest,ytest],axis = 1)

In [18]:
print(Train.info())
print(Train.columns)
Train.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1449 entries, 0 to 1448
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   valid_time         1449 non-null   datetime64[s]
 1   wind300_max        1449 non-null   float64      
 2   wind300_min        1449 non-null   float64      
 3   wind300_mean       1449 non-null   float64      
 4   wind400_max        1449 non-null   float64      
 5   wind400_min        1449 non-null   float64      
 6   wind400_mean       1449 non-null   float64      
 7   temp300_max        1449 non-null   float64      
 8   temp300_min        1449 non-null   float64      
 9   temp300_mean       1449 non-null   float64      
 10  temp400_max        1449 non-null   float64      
 11  temp400_min        1449 non-null   float64      
 12  temp400_mean       1449 non-null   float64      
 13  z300_min           1449 non-null   float64      
 14  z300_max           1449 non-null   

,valid_time,wind300_max,wind300_min,wind300_mean,wind400_max,wind400_min,wind400_mean,temp300_max,temp300_min,temp300_mean,...,r400_mean,tcc_max,tcc_min,tcc_mean,daily_rainfall,host,totmembers,tothired,prev_day_rainfall,%_mem_succs
0,1995-05-11,5.666456,2.984347,4.294215,3.168471,1.435653,2.578355,246.19069,245.05974,245.411988,...,0.480584,0.870941,0.040680,0.561104,0.000242,2,23,2,0.000558,0.304348
1,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.470559,0.877533,0.063324,0.600616,0.000326,2,13,7,0.000242,0.153846
2,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.470559,0.877533,0.063324,0.600616,0.000326,2,6,0,0.000242,0.500000
3,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.470559,0.877533,0.063324,0.600616,0.000326,2,12,4,0.000242,0.666667
4,1995-05-12,6.885866,3.284288,5.473119,4.154729,3.141184,3.486980,247.03448,246.01244,246.391160,...,0.470559,0.877533,0.063324,0.600616,0.000326,1,11,15,0.000242,0.000000
5,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.308656,0.970734,0.707428,0.898720,0.001482,2,19,12,0.000606,0.473684
6,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.308656,0.970734,0.707428,0.898720,0.001482,2,10,3,0.000606,0.100000
7,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.308656,0.970734,0.707428,0.898720,0.001482,2,8,2,0.000606,0.375000
8,1995-05-14,5.135309,2.575506,4.099943,7.995140,1.383607,4.692359,247.22226,245.95332,246.744780,...,0.308656,0.970734,0.707428,0.898720,0.001482,2,20,14,0.000606,0.400000
9,1995-05-15,7.593754,4.862681,5.913409,7.640121,0.979840,5.055960,246.86212,245.58328,246.334735,...,0.378795,0.998657,0.785858,0.921677,0.002844,1,12,9,0.001482,0.166667


In [19]:
print(Test.info())
print(Test.columns)
Test.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 162 entries, 1449 to 1610
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   valid_time         162 non-null    datetime64[s]
 1   wind300_max        162 non-null    float64      
 2   wind300_min        162 non-null    float64      
 3   wind300_mean       162 non-null    float64      
 4   wind400_max        162 non-null    float64      
 5   wind400_min        162 non-null    float64      
 6   wind400_mean       162 non-null    float64      
 7   temp300_max        162 non-null    float64      
 8   temp300_min        162 non-null    float64      
 9   temp300_mean       162 non-null    float64      
 10  temp400_max        162 non-null    float64      
 11  temp400_min        162 non-null    float64      
 12  temp400_mean       162 non-null    float64      
 13  z300_min           162 non-null    float64      
 14  z300_max           162 non-null  

,valid_time,wind300_max,wind300_min,wind300_mean,wind400_max,wind400_min,wind400_mean,temp300_max,temp300_min,temp300_mean,...,r400_mean,tcc_max,tcc_min,tcc_mean,daily_rainfall,host,totmembers,tothired,prev_day_rainfall,%_mem_succs
1449,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,19,41,0.001058,0.789474
1450,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,4,6,0.001058,0.750000
1451,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,9,7,0.001058,0.222222
1452,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,18,50,0.001058,0.777778
1453,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,15,23,0.001058,0.800000
1454,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,18,35,0.001058,0.611111
1455,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,11,14,0.001058,0.636364
1456,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,11,14,0.001058,0.545455
1457,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,4,10,0.001058,0.500000
1458,2022-05-12,18.760017,16.275662,17.313162,10.338636,6.499078,8.625193,248.58334,247.89508,248.157053,...,0.355733,0.933472,0.680695,0.807999,0.003124,1,7,7,0.001058,0.714286


In [21]:
try:
    Train.to_csv('Data/Training_Data.csv')
    Test.to_csv('Data/Testing_Data.csv')
except: 
    raise('BufferError')